# الدرس الاول: بناء الوكلاء الاذكياء في معمارية LangChain الحديثة

## المقدمة والاهداف التعليمية
في هذا الدفتر، سنتعلم كيفية بناء وكيل ذكي (AI Agent) قادر على اتخاذ القرارات واستدعاء الادوات البرمجية للاجابة عن الاسئلة.

## ما الجديد في LangChain 1.x مقارنة بالاصدارات السابقة؟
1. تم استبدال الفئة القديمة `AgentExecutor` المعتمدة على سلاسل التنفيذ الخطية بالدالة الحديثة `create_agent` المعتمدة داخليا على محرك الرسوم البيانية `LangGraph` (`CompiledStateGraph`).
2. اصبح الوكيل يعتمد كليا على بروتوكول استدعاء الادوات الاصلي (Native Tool Calling) الذي توفره نماذج المحادثة بدلا من تقنيات التحليل النصي اليدوي (Regex parsing / MRKL).
3. يرجع الوكيل كائنا من نوع `CompiledStateGraph` يتيح تتبع الحالات (State Management)، التراجع الزمني، والدعم الكامل للتنفيذ اللاتزامني والتدفق المستمر.

## الخطوة 1: التحقق من بيئة العمل واصدار المكتبات
نقوم بالتحقق من اصدار مكتبة `langchain` للتأكد من اننا نستخدم المعمارية الحديثة.

In [ ]:
import langchain
import langchain_core

print("LangChain Version:", langchain.__version__)
print("LangChain Core Version:", langchain_core.__version__)

## الخطوة 2: تحميل المتغيرات البيئية وتهيئة مفاتيح الربط
نستخدم `dotenv` لقراءة متغيرات البيئة بصورة آمنة دون تضمين المفاتيح السرية في الكود.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY", "")

## الخطوة 3: تهيئة نموذج اللغة وتحديد الادوات
نقوم بتهيئة نموذج الدردشة عبر `ChatGroq` مع تحديد درجة الحرارة بـ 0 لضمان دقة استدعاء الادوات.
بعد ذلك نعرف دالة بايثون بسيطة مع التوثيق الكامل وتحديد انواع المدخلات والمخرجات (Type Hints)؛ حيث يعتمد النموذج على النص التوثيقي (Docstring) لفهم متى وكيف يستدعي الاداة.

In [ ]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent

# تهيئة النموذج
model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

# تعريف اداة الاستعلام عن حالة الطقس
def get_weather(city: str) -> str:
    """Get current weather conditions for a given city name."""
    city_normalized = city.strip().title()
    # محاكاة لبيانات الطقس
    weather_database = {
        "New York": "Sunny, 72 F, Humidity: 45%",
        "London": "Cloudy with light rain, 60 F, Humidity: 80%",
        "Cairo": "Clear and warm, 86 F, Humidity: 35%",
        "Tokyo": "Partly cloudy, 68 F, Humidity: 55%"
    }
    return weather_database.get(
        city_normalized, 
        f"The weather in {city_normalized} is clear and pleasant."
    )

## الخطوة 4: انشاء الوكيل عبر `create_agent`
نستخدم المعمارية الحديثة `create_agent` لتجميع النموذج والادوات والتوجيهات العامة (System Prompt).
تنتج هذه الدالة كائنا قابلا للتشغيل (`CompiledStateGraph`) يتوافق مع واجهة LangChain الموحدة.

In [ ]:
# انشاء الوكيل المعتمد على الرسم البياني للحالة
agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt="You are a professional assistant. When asked about weather in any city, use the get_weather tool to provide accurate answers."
)

print("Agent Type:", type(agent))

## الخطوة 5: استدعاء الوكيل وفحص المخرجات
نقوم بتمرير رسالة المستخدم الى الوكيل عبر قاموس يحتوي على قائمة `messages`.
يقوم الوكيل بتحديد الاداة المناسبة، تنفيذها، ثم صياغة الاجابة النهائية بناء على ناتج الاداة.

In [ ]:
# تنفيذ استعلام المستخدم
response = agent.invoke({
    "messages": [
        ("user", "What is the current weather in New York and Cairo?")
    ]
})

# طباعة نص الرد النهائي
final_message = response["messages"][-1]
print("Final Response:")
print(final_message.content)

## الخطوة 6: فحص سجل الرسائل وخطوات اتخاذ القرار
الميزة الكبرى في المعمارية الحديثة هي الشفافية؛ حيث يمكننا مراجعة قائمة الرسائل بالكامل لرؤية:
1. رسالة المستخدم الاصلية (HumanMessage).
2. استدعاء النموذج للاداة (AIMessage with tool_calls).
3. ناتج تنفيذ الاداة (ToolMessage).
4. الرد التجميعي النهائي للنموذج.

In [ ]:
print("--- Execution History ---")
for idx, msg in enumerate(response["messages"]):
    role = getattr(msg, "type", type(msg).__name__)
    print(f"Step {idx + 1}: [{role}]")
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"   Tool Calls: {msg.tool_calls}")
    elif hasattr(msg, "content"):
        content_preview = str(msg.content)[:120]
        print(f"   Content: {content_preview}...")